In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Load Data
data_path = r"C:\Users\ACER\tinyml-iot-security\data\synthetic_L64.csv"
df = pd.read_csv(data_path)

X_raw = df.drop(columns=['label']).values
y_raw = df['label'].values

# 2. Reshape into raw time-series matrices (Samples, 64 Timesteps, 5 Channels)
# Note: The original dataset text template notes 16 channels, but our generated layout maps to 5 channels
num_samples = X_raw.shape[0]
timesteps = 64
num_channels = 5
X_reshaped = X_raw.reshape(num_samples, timesteps, num_channels)

# 3. Stratified Split (70% Train, 15% Validation, 15% Test)
# First split out Train (70%) and a temporary holding set (30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X_reshaped, y_raw, test_size=0.30, random_state=42, stratify=y_raw
)

# Split the remaining 30% equally into Validation (15%) and Test (15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

# 4. Standard Scaling (Normalization)
# Flatten to 2D to fit scaler, then restore original dimensions
scaler = StandardScaler()

X_train_flat = X_train.reshape(-1, num_channels)
X_train_scaled = scaler.fit_transform(X_train_flat).reshape(X_train.shape)

X_val_flat = X_val.reshape(-1, num_channels)
X_val_scaled = scaler.transform(X_val_flat).reshape(X_val.shape)

X_test_flat = X_test.reshape(-1, num_channels)
X_test_scaled = scaler.transform(X_test_flat).reshape(X_test.shape)

# 5. Verify Split Integrity
print("=== PREPROCESSING MATRIX REGISTRATION ===")
print(f"X_train shape: {X_train_scaled.shape} | y_train shape: {y_train.shape}")
print(f"X_val shape:   {X_val_scaled.shape}  | y_val shape:   {y_val.shape}")
print(f"X_test shape:  {X_test_scaled.shape}  | y_test shape:  {y_test.shape}")

print("\nProportional Class Representation (Train vs Val):")
print(f"Train labels balance: {np.bincount(y_train) / len(y_train)}")
print(f"Val labels balance:   {np.bincount(y_val) / len(y_val)}")


=== PREPROCESSING MATRIX REGISTRATION ===
X_train shape: (1400, 64, 5) | y_train shape: (1400,)
X_val shape:   (300, 64, 5)  | y_val shape:   (300,)
X_test shape:  (300, 64, 5)  | y_test shape:  (300,)

Proportional Class Representation (Train vs Val):
Train labels balance: [0.61071429 0.19714286 0.19214286]
Val labels balance:   [0.61       0.19666667 0.19333333]
